In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv


In [2]:
!pip install -q transformers datasets accelerate evaluate wandb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


In [3]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import wandb

from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [4]:
wandb.login()

wandb.init(
    project="twitter-sentiment-modern",
    config={
        "model": "roberta-base-bilstm-attention",
        "dataset": "Sentiment140",
        "epochs": 2,
        "batch_size": 32,
        "lr": 2e-5
    }
)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Find your API key here: https://wandb.ai/authorize?ref=models
wandb: Paste an API key from your profile and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: 23bec032 (23bec032-iiit-dharwad) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [5]:
from datasets import load_dataset

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv",
    column_names=["sentiment","id","date","flag","user","text"],
    encoding="latin-1"   # ✅ FIX
)

dataset = dataset["train"]

def preprocess(example):
    label = 1 if example["sentiment"] == 4 else 0
    return {"text": example["text"], "label": label}

dataset = dataset.map(preprocess)

dataset = dataset.remove_columns(["sentiment","id","date","flag","user"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/1600000 [00:00<?, ? examples/s]

In [6]:
dataset = dataset.train_test_split(test_size=0.1)

train_dataset = dataset["train"]
test_dataset = dataset["test"]

In [7]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding=False,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map:   0%|          | 0/1440000 [00:00<?, ? examples/s]

Map:   0%|          | 0/160000 [00:00<?, ? examples/s]

In [8]:
class Attention(nn.Module):

    def __init__(self, hidden_dim):
        super().__init__()
        self.attention = nn.Linear(hidden_dim, 1)

    def forward(self, x):

        weights = torch.softmax(self.attention(x), dim=1)

        context = torch.sum(weights * x, dim=1)

        return context


class SentimentModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.roberta = AutoModel.from_pretrained("roberta-base")

        self.lstm = nn.LSTM(
            input_size=768,
            hidden_size=256,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = Attention(512)

        self.dropout = nn.Dropout(0.3)

        self.classifier = nn.Linear(512, 2)

    def forward(self, input_ids, attention_mask, labels=None):

        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        sequence_output = outputs.last_hidden_state

        lstm_output, _ = self.lstm(sequence_output)

        attention_output = self.attention(lstm_output)

        logits = self.classifier(self.dropout(attention_output))

        loss = None

        if labels is not None:
            loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
            loss = loss_fn(logits, labels)

        return {"loss": loss, "logits": logits}

In [9]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    preds = np.argmax(logits, axis=1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        preds,
        average="binary"
    )

    acc = accuracy_score(labels, preds)

    roc = roc_auc_score(labels, logits[:,1])

    return {
        "accuracy": acc,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "roc_auc": roc
    }

In [10]:
from transformers import TrainingArguments

training_args = TrainingArguments(

    output_dir="./results",

    learning_rate=2e-5,

    per_device_train_batch_size=32,

    per_device_eval_batch_size=32,

    num_train_epochs=2,

    weight_decay=0.01,

    eval_strategy="epoch",      # ✅ FIXED

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="f1",

    logging_dir="./logs",

    report_to="wandb",

    fp16=True,

    save_total_limit=2
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [11]:
model = SentimentModel().to(device)

data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset,

    data_collator=data_collator,

    compute_metrics=compute_metrics,

    processing_class=tokenizer   # ✅ FIXED
)

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [12]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc
1,0.374152,0.371264,0.892750,0.903949,0.879567,0.891591,0.959614
2,0.344745,0.364790,0.898250,0.904088,0.891670,0.897836,0.963021


TrainOutput(global_step=90000, training_loss=0.36983865509033204, metrics={'train_runtime': 14565.1348, 'train_samples_per_second': 197.732, 'train_steps_per_second': 6.179, 'total_flos': 0.0, 'train_loss': 0.36983865509033204, 'epoch': 2.0})

In [13]:
trainer.save_model("./best_model")

tokenizer.save_pretrained("./best_model")

wandb.save("./best_model/*")

wandb: WARNING Symlinked 4 files into the W&B run directory; call wandb.save again to sync new files.


['/kaggle/working/wandb/run-20260226_163042-02nsgh9m/files/best_model/model.safetensors',
 '/kaggle/working/wandb/run-20260226_163042-02nsgh9m/files/best_model/tokenizer.json',
 '/kaggle/working/wandb/run-20260226_163042-02nsgh9m/files/best_model/tokenizer_config.json',
 '/kaggle/working/wandb/run-20260226_163042-02nsgh9m/files/best_model/training_args.bin']

In [14]:
results = trainer.evaluate()

print(results)

wandb.log(results)

{'eval_loss': 0.3647904396057129, 'eval_accuracy': 0.89825, 'eval_precision': 0.904088467614534, 'eval_recall': 0.8916698866964987, 'eval_f1': 0.8978362367588735, 'eval_roc_auc': 0.9630211725809377, 'eval_runtime': 257.8248, 'eval_samples_per_second': 620.576, 'eval_steps_per_second': 19.393, 'epoch': 2.0}


In [15]:
def predict(text):

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    ).to(device)

    with torch.no_grad():

        outputs = model(**inputs)

        logits = outputs["logits"]

        pred = torch.argmax(logits, dim=1).item()

    return "Positive" if pred == 1 else "Negative"


print(predict("I love this movie"))
print(predict("This is terrible"))

Positive
Negative
